# 요약 방식 비교: summarize_rfp vs search_rfp

평가셋 `type == "summary"` 문항으로 네 방식을 비교합니다.

| 방식 | 내용 | 요약 안 LLM 호출 |
|---|---|---|
| A | 에이전트 + `summarize_rfp` (항목별 검색 → 항목별 추출) | 9번 |
| B | 에이전트 + `search_rfp` 만 (기존 방식, LLM이 알아서 검색해서 요약) | - |
| C | 항목별 검색 → 청크를 모아 한 번에 추출 | 1번 |
| D | 항목을 4묶음으로 나눠 묶음별 검색 → 추출 (A와 C의 중간) | 4번 |

C·D는 A와 같은 검색(의미 검색 + 예산·기간·계약방법 키워드 청크)과 같은 공고 정보 보완을 씁니다.
지표: 11개 항목 점수(LLM 채점), 지어낸 항목, 놓친 항목, 시간, LLM 호출 수, 토큰.
결과는 `eval/results/` 에 저장됩니다 (gitignore).

In [6]:
import importlib
import json
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage

from rfp_agent.agent.graph import build_graph
from rfp_agent.agent.tools.search_rfp import search_rfp
from rfp_agent.agent.trace import LLMCallRecorder
from rfp_agent.config import PROJECT_ROOT, load_config
from rfp_agent.llm import get_llm
from rfp_agent.retrieval.retriever import retrieve

S = importlib.import_module("rfp_agent.agent.tools.summarize_rfp")

# ===== 실험 설정 =====
CONFIG_NAME = "configs/ollama.yaml"   # 요약할 때 쓸 LLM (configs/default.yaml 이면 OpenAI)
JUDGE = "ollama"                      # 채점 LLM: "openai"(gpt-4o-mini) 또는 "ollama"(gemma4:12b). 채점 셀에서 스키마 강제로 만듦
VARIANTS_TO_RUN = ["A", "B", "C", "D"]
LOAD_RESULTS = None                   # 예: "eval/results/summary_compare_..._ollama.jsonl" → 실행 건너뛰고 채점만
# 한 번에 넣을 청크 글자 수 상한. gemma 8k 컨텍스트면 9,000자 정도가 안전, OpenAI 는 30,000 까지 OK
MAX_CONTEXT_CHARS = 9000 if "ollama" in CONFIG_NAME else 30000
# ====================

config = load_config(CONFIG_NAME)
config["agent"] = {**config.get("agent", {}), "show_trace": False}  # 실험에서는 답변에 로그를 붙이지 않음
ollama_cfg = load_config("configs/ollama.yaml")["llm"]


def json_llm(provider):
    """C·D 추출용: JSON 만 내도록 강제한 LLM. ollama 는 format=json, openai 는 json_object 모드."""
    if provider == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(base_url=ollama_cfg.get("base_url", "http://localhost:11434"),
                          model=ollama_cfg["model"], temperature=0,
                          num_ctx=ollama_cfg.get("num_ctx", 8192), format="json", reasoning=False)
    return get_llm({"llm": {"provider": "openai", "model": "gpt-4o-mini", "temperature": 0}}).bind(
        response_format={"type": "json_object"})


extract_llm = json_llm(config["llm"]["provider"])  # C·D 추출용 (요약 LLM 과 같은 모델)
RESULTS_DIR = PROJECT_ROOT / "eval/results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
rows = [json.loads(line) for line in open(PROJECT_ROOT / "eval/questions.jsonl", encoding="utf-8") if line.strip()]
summary_qs = [r for r in rows if r["type"] == "summary"]
for r in summary_qs:
    r["reference"] = r.get("answer_facts") or r.get("answer") or {}
print(len(summary_qs), "문항:", [r["id"] for r in summary_qs])

8 문항: ['jm-06', 'yg-08', 'yg-09', 'yg-10', 'wt-02', 'yj-06', 'yg-18', 'yg-19']


## 방식 정의
각 함수는 `(답변, LLM 호출 기록)` 을 돌려줍니다. 시간은 바깥에서 잽니다.

In [3]:
graph_a = build_graph(config, tools=[S.summarize_rfp])
graph_b = build_graph(config, tools=[search_rfp])


def run_agent(graph, question):
    recorder = LLMCallRecorder()
    result = graph.invoke({"messages": [("user", question)]}, config={"callbacks": [recorder]})
    return result["messages"][-1].content, recorder.snapshot()


def parse_json(text):
    """코드블록·앞뒤 말이 붙어도 첫 번째 {...} 를 찾아 읽습니다. 실패하면 {}."""
    match = re.search(r"\{.*\}", text or "", re.S)
    if not match:
        return {}
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}
    return value if isinstance(value, dict) else {}


def as_text(value):
    if isinstance(value, list):
        return "\n".join(str(v) for v in value)
    return str(value).strip() if value not in (None, "") else S.NOT_FOUND


GROUP_PROMPT = """당신은 RFP(제안요청서)에서 정보를 정확히 옮겨 적는 도우미입니다.
[공고 정보]와 [문서 발췌]만 보고 아래 항목을 JSON 객체로 채우세요. 키는 항목 이름 그대로 씁니다.
항목과 설명:
{fields}
- 발췌에 있는 내용만 쓰고 금액·날짜·비율은 문서 표기 그대로 옮기세요.
- [공고 정보]는 제공받은 공고 목록의 값이라 근거로 써도 됩니다.
- 근거를 찾을 수 없는 항목은 값을 정확히 "{not_found}" 로 쓰세요. 추측하지 마세요.
- 다른 공고문을 참조하라고 되어 있으면 그 사실을 쓰세요.
- 값은 문자열이며 여러 개면 줄바꿈으로 구분합니다. JSON 외에 다른 말은 쓰지 마세요."""

LLM_FIELDS = [f for f in S.SUMMARY_FIELDS if f not in S.METADATA_FIELDS]  # 발주기관·사업명은 메타데이터
GROUPS = {
    "C": [LLM_FIELDS],
    "D": [["예산", "사업기간", "계약방법"], ["주요 요구사항", "참가 자격"],
          ["평가", "제출 마감"], ["하자보수", "공동수급·하도급"]],
}


def group_messages(fields, record):
    per_field = max(1, S.CHUNKS_PER_FIELD if len(fields) <= 3 else 2)
    search_config = {**config, "retrieval": {**config["retrieval"], "top_k": per_field}}
    groups = []
    for field in fields:
        query = S.SUMMARY_FIELDS[field][0]
        groups += [S._keyword_chunks(field, record["doc_id"], config),
                   retrieve(query, search_config, filters={"doc_id": record["doc_id"]})]
    chunks, size = [], 0
    for doc in S._merge(*groups):  # 글자 수 상한까지만 넣음 (넘으면 뒤 청크는 버림)
        if size + len(doc.page_content) > MAX_CONTEXT_CHARS:
            break
        chunks.append(doc)
        size += len(doc.page_content)
    hint = "".join(S._metadata_hint(f, record) for f in fields)
    spec = "\n".join(f"- {f}: {S.SUMMARY_FIELDS[f][1]}" for f in fields)
    return [SystemMessage(GROUP_PROMPT.format(fields=spec, not_found=S.NOT_FOUND)),
            HumanMessage(f"{hint}[문서 발췌]\n{S._format_chunks(chunks)}")]


def run_grouped(q, groups):
    recorder = LLMCallRecorder()
    candidates = S.find_rfp(rfp_name_of(q), S.load_catalog(config))
    if len(candidates) != 1:
        return f"공고 특정 실패: 후보 {len(candidates)}건", []
    record = candidates[0]
    fields = {f: S._first(record, S.METADATA_FIELDS[f]) or S.NOT_FOUND for f in S.METADATA_FIELDS}
    answers = extract_llm.batch([group_messages(g, record) for g in groups],
                                config={"callbacks": [recorder], "max_concurrency": 4})
    for group, answer in zip(groups, answers):
        values = parse_json(answer.content)
        for f in group:
            text = as_text(values.get(f))
            fields[f] = S.NOT_FOUND if S.NOT_FOUND in text else text
    for f in S.SUMMARY_FIELDS:
        if fields.get(f, S.NOT_FOUND) == S.NOT_FOUND:
            fields[f] = S._fallback(f, record) or S.NOT_FOUND
    ordered = {f: fields[f] for f in S.SUMMARY_FIELDS}
    return S.format_summary(record, ordered), recorder.snapshot()


def rfp_name_of(q):
    # C·D 는 에이전트가 없으니 정답 문서의 사업명을 그대로 넣음 (공고 찾기는 A와 같은 함수)
    doc_id = q["answer_doc_ids"][0]
    record = next((r for r in S.load_catalog(config) if r["doc_id"] == doc_id), None)
    return S._first(record, S.METADATA_FIELDS["사업명"]) if record else q["question"]


VARIANTS = {
    "A": lambda q: run_agent(graph_a, q["question"]),
    "B": lambda q: run_agent(graph_b, q["question"]),
    "C": lambda q: run_grouped(q, GROUPS["C"]),
    "D": lambda q: run_grouped(q, GROUPS["D"]),
}

## 실행
문항 × 방식마다 1번씩 돌립니다. `LOAD_RESULTS` 를 지정하면 저장된 결과를 불러와 채점만 다시 합니다.

In [4]:
if LOAD_RESULTS:
    out = PROJECT_ROOT / LOAD_RESULTS
    results = [json.loads(line) for line in out.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("불러옴:", out, len(results), "건")
else:
    results = []
    for q in summary_qs:
        for name in VARIANTS_TO_RUN:
            start = time.perf_counter()
            try:
                answer, calls = VARIANTS[name](q)
            except Exception as exc:  # noqa: BLE001  연결 오류 등은 기록하고 계속
                answer, calls = f"ERROR: {exc!r}", []
            sec = time.perf_counter() - start
            tokens = sum(((c.get("tokens") or {}).get("input_tokens") or 0)
                         + ((c.get("tokens") or {}).get("output_tokens") or 0) for c in calls)
            results.append({"id": q["id"], "variant": name, "answer": answer, "sec": round(sec, 1),
                            "llm_calls": len(calls), "tokens": tokens,
                            "tool_calls": [t["name"] for c in calls for t in c.get("tool_calls", [])]})
            print(f"{q['id']:6} {name}  {sec:6.1f}s  calls={len(calls):2}  tokens={tokens}")

    stamp = datetime.now().strftime("%Y%m%d_%H%M")
    out = RESULTS_DIR / f"summary_compare_{stamp}_{Path(CONFIG_NAME).stem}.jsonl"
    out.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8")
    print("저장:", out)

jm-06  A    50.2s  calls=11  tokens=23200
jm-06  B    16.0s  calls= 2  tokens=3906
jm-06  C    40.8s  calls= 1  tokens=8006
jm-06  D   164.4s  calls= 4  tokens=25305
yg-08  A    56.4s  calls=11  tokens=25016
yg-08  B    14.8s  calls= 2  tokens=4143
yg-08  C    35.4s  calls= 1  tokens=8190
yg-08  D   152.8s  calls= 4  tokens=25993
yg-09  A    33.2s  calls=11  tokens=19858
yg-09  B    14.6s  calls= 2  tokens=4017
yg-09  C    31.2s  calls= 1  tokens=7502
yg-09  D   108.8s  calls= 4  tokens=21389
yg-10  A    38.7s  calls=11  tokens=22483
yg-10  B    20.5s  calls= 2  tokens=4988
yg-10  C    36.5s  calls= 1  tokens=7817
yg-10  D   129.8s  calls= 4  tokens=23298
wt-02  A    45.9s  calls=11  tokens=23539
wt-02  B    11.7s  calls= 2  tokens=4568
wt-02  C    37.6s  calls= 1  tokens=8191
wt-02  D   146.4s  calls= 4  tokens=24802
yj-06  A    58.5s  calls=11  tokens=24311
yj-06  B    11.4s  calls= 2  tokens=3837
yj-06  C    37.2s  calls= 1  tokens=8190
yj-06  D   170.5s  calls= 4  tokens=27535
yg-1

## 채점 (LLM-as-judge)

항목마다 하나로 판정합니다.
- `correct` 정답과 핵심이 같음 / `partial` 일부만 맞음 / `wrong` 틀림
- `missing` 정답엔 있는데 "문서에 명시 없음" 이나 누락
- `hallucinated` 정답에 없는(null·키 없음) 항목인데 구체적 내용을 씀, 또는 문서에 없는 금액·날짜를 지어냄
- `ok_not_found` 정답에도 없고 답변도 "명시 없음"

판정이 비거나 11개 항목이 다 안 오면 최대 3번 다시 요청합니다. 그래도 실패한 건 `judge_ok=False` 로 표시하고 평균에서 뺍니다.
이 셀만 다시 돌리면 실패한 건만 다시 채점합니다.

In [19]:
VERDICTS = {"correct", "partial", "wrong", "missing", "hallucinated", "ok_not_found"}
SCORE = {"correct": 1, "partial": 0.5, "ok_not_found": 1}

# 채점 결과 형식을 스키마로 강제: 11개 항목이 꼭 있고, 값은 VERDICTS 중 하나만 나올 수 있음
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {f: {"type": "string", "enum": sorted(VERDICTS)} for f in S.SUMMARY_FIELDS},
    "required": list(S.SUMMARY_FIELDS),
    "additionalProperties": False,
}
if JUDGE == "ollama":
    from langchain_ollama import ChatOllama
    judge_llm = ChatOllama(base_url=ollama_cfg.get("base_url", "http://localhost:11434"),
                           model=ollama_cfg["model"], temperature=0,
                           num_ctx=ollama_cfg.get("num_ctx", 8192),
                           format=JUDGE_SCHEMA, reasoning=False)
else:
    judge_llm = get_llm({"llm": {"provider": "openai", "model": "gpt-4o-mini", "temperature": 0}}).bind(
        response_format={"type": "json_schema",
                         "json_schema": {"name": "verdicts", "strict": True, "schema": JUDGE_SCHEMA}})
JUDGE_PROMPT = """RFP 요약 답변을 정답과 항목별로 비교해 채점합니다.
판정값: correct, partial, wrong, missing, hallucinated, ok_not_found
- 정답 값이 null 이거나 키가 없으면 문서에 없는 항목입니다. 답변이 "명시 없음"이면 ok_not_found, 구체적 내용을 쓰면 hallucinated.
- 정답에 값이 있는데 답변이 "명시 없음"이거나 빠졌으면 missing.
- 표현이 달라도 핵심 사실(금액·기간·방식·비율)이 같으면 correct. 일부만 맞으면 partial.
- 정답에 없는 금액·날짜를 지어냈으면 hallucinated.
- "문서에 명시 없음 (공고 목록상 …)" 처럼 괄호로 공고 목록 값을 덧붙인 답은 "명시 없음"으로 보고 판정하세요. 괄호 안 값은 참고 정보라 지어낸 것으로 보지 않습니다.
다음은 답변에 들어가면 안 되는 내용입니다. 해당 항목에 들어가 있으면 그 항목을 hallucinated 로 판정하세요: {must_not}
아래 11개 항목 이름을 키로, 판정값을 값으로 하는 JSON 객체 하나만 출력하세요.
항목: {fields}"""


def judge(q, answer, tries=3):
    messages = [
        SystemMessage(JUDGE_PROMPT.format(must_not=q.get("must_not") or "없음",
                                          fields=", ".join(S.SUMMARY_FIELDS))),
        HumanMessage(f"[정답]\n{json.dumps(q['reference'], ensure_ascii=False)}\n\n[답변]\n{answer}"),
    ]
    raw, verdicts = "", {}
    for _ in range(tries):
        raw = judge_llm.invoke(messages).content
        verdicts = {k: v for k, v in parse_json(raw).items() if k in S.SUMMARY_FIELDS and v in VERDICTS}
        if len(verdicts) == len(S.SUMMARY_FIELDS):
            return verdicts, raw, True
    return verdicts, raw, False


by_id = {q["id"]: q for q in summary_qs}
for r in results:
    if r.get("judge_ok"):
        continue  # 이미 채점된 건 건너뜀
    if r["answer"].startswith("ERROR") or not r["answer"].strip():
        r.update(verdicts={}, judge_raw="", judge_ok=False, empty_answer=True)
    else:
        r["verdicts"], r["judge_raw"], r["judge_ok"] = judge(by_id[r["id"]], r["answer"])
        r["empty_answer"] = False
    r["score"] = sum(SCORE.get(r["verdicts"].get(f), 0) for f in S.SUMMARY_FIELDS)
    r["hallucinated"] = sum(v == "hallucinated" for v in r["verdicts"].values())
    r["missing"] = sum(v == "missing" for v in r["verdicts"].values())
    print(f"{r['id']:6} {r['variant']}  judge_ok={r['judge_ok']}  score={r['score']}")
judged_path = out.with_name(out.stem.removesuffix("_judged") + "_judged.jsonl")
judged_path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8")
print("채점 실패:", sum(not r["judge_ok"] for r in results), "/", len(results), "→ 저장:", judged_path)

jm-06  D  judge_ok=False  score=8.0
채점 실패: 1 / 32 → 저장: /Users/codeit/dev/rfp-agent-part3-team1/eval/results/summary_compare_20261123_1700_ollama_judged.jsonl


## 결과
평균은 채점에 성공한 건만으로 냅니다. `judged` 가 방식마다 다르면 비교할 때 주의하세요.

In [11]:
df = pd.DataFrame(results)
ok = df[df.judge_ok]
summary = ok.groupby("variant").agg(
    judged=("id", "count"), score_of_11=("score", "mean"),
    hallucinated=("hallucinated", "sum"), missing=("missing", "sum"),
).join(df.groupby("variant").agg(
    empty_or_error=("empty_answer", "sum"),
    sec=("sec", "mean"), llm_calls=("llm_calls", "mean"), tokens=("tokens", "mean"),
)).round(2)
summary

,judged,score_of_11,hallucinated,missing,empty_or_error,sec,llm_calls,tokens
variant,,,,,,,,
A,8,8.69,4,6,0,46.76,11.0,23122.75
B,8,5.69,2,37,0,14.71,2.0,4202.00
C,8,5.31,7,31,0,36.58,1.0,8022.25
D,7,8.14,6,5,0,136.60,4.0,24325.88


In [12]:
# 모든 방식이 채점된 문항끼리만 비교 (가장 공정한 비교)
common = ok.groupby("id").variant.nunique()
common_ids = common[common == ok.variant.nunique()].index
print("공통 문항:", list(common_ids))
ok[ok.id.isin(common_ids)].groupby("variant").score.mean().round(2)

공통 문항: ['wt-02', 'yg-08', 'yg-09', 'yg-10', 'yg-18', 'yg-19', 'yj-06']


variant
A    8.57
B    5.71
C    4.93
D    8.14
Name: score, dtype: float64

In [13]:
# 문항별 점수 (빈칸 = 채점 실패)
ok.pivot(index="id", columns="variant", values="score")

variant,A,B,C,D
id,,,,
jm-06,9.5,5.5,8.0,NaN
wt-02,8.5,3.5,3.0,8.5
yg-08,9.0,7.0,3.0,8.5
yg-09,8.0,6.5,7.5,7.5
yg-10,7.5,5.5,8.5,7.0
yg-18,8.0,8.0,4.0,7.5
yg-19,9.0,4.0,4.5,7.0
yj-06,10.0,5.5,4.0,11.0


In [14]:
# 항목별 정확도 (어느 항목에서 방식 차이가 나는지)
field_rows = [{"variant": r["variant"], "field": f, "ok": SCORE.get(r["verdicts"].get(f), 0)}
              for r in results if r["judge_ok"] for f in S.SUMMARY_FIELDS]
pd.DataFrame(field_rows).pivot_table(index="field", columns="variant", values="ok", aggfunc="mean").round(2)

variant,A,B,C,D
field,,,,
계약방법,0.88,0.69,0.44,0.93
공동수급·하도급,0.69,0.00,0.38,0.71
발주기관,0.94,0.88,0.94,0.93
사업기간,0.94,0.88,0.38,0.79
사업명,1.00,1.00,1.00,1.00
예산,0.88,0.75,0.88,0.86
제출 마감,0.38,0.38,0.12,0.43
주요 요구사항,0.75,0.75,0.38,0.64
참가 자격,0.75,0.12,0.19,0.79


In [15]:
# 도구 사용 확인: A 는 summarize_rfp, B 는 search_rfp 만 불러야 정상
df[["id", "variant", "tool_calls", "sec", "llm_calls"]]

,id,variant,tool_calls,sec,llm_calls
0,jm-06,A,[summarize_rfp],50.2,11
1,jm-06,B,[search_rfp],16.0,2
2,jm-06,C,[],40.8,1
3,jm-06,D,[],164.4,4
4,yg-08,A,[summarize_rfp],56.4,11
5,yg-08,B,[search_rfp],14.8,2
6,yg-08,C,[],35.4,1
7,yg-08,D,[],152.8,4
8,yg-09,A,[summarize_rfp],33.2,11
9,yg-09,B,[search_rfp],14.6,2


In [16]:
# 답변 하나 직접 보기
from IPython.display import Markdown
pick = df[(df.id == summary_qs[0]["id"]) & (df.variant == "A")].iloc[0]
print(pick.verdicts)
Markdown(pick.answer)

{'발주기관': 'correct', '사업명': 'correct', '예산': 'correct', '사업기간': 'correct', '계약방법': 'correct', '주요 요구사항': 'partial', '참가 자격': 'correct', '평가': 'correct', '제출 마감': 'wrong', '하자보수': 'correct', '공동수급·하도급': 'correct'}


JST 공유대학(원) xAPI 기반 LRS시스템 구축 사업의 요약 내용입니다.

### 📄 JST 공유대학(원) xAPI기반 LRS시스템 구축

**출처 문서:** 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp

- **발주기관**: 전북대학교
- **사업명**: JST 공유대학(원) xAPI기반 LRS시스템 구축
- **예산**: 금116,000,000원(금일억일천육백만원, 부가세 포함)
- **사업기간**: 계약체결일로부터 75일까지
- **계약방법**
  - 제한(총액)경쟁입찰(협상에 의한 계약)
  - 협상에 의한 계약
  - 제한(총액)경쟁입찰, 협상에 의한 계약
- **주요 요구사항**
  - 「행정기관 및 공공기관 정보시스템 구축·운영 지침」 준수 및 개방형 기술 구현
  - 소프트웨어개발 보안 가이드 준수 및 시큐어코딩을 통한 보안취약점 제거
  - SW사업정보 데이터 작성 및 제출(기능점수 측정 전문가 포함)
  - 응용프로그램과 DB 간 일관성 유지 및 DB 최적화
  - 기존 정보시스템 데이터의 정확성 점검 및 매핑을 통한 데이터 이관
  - 운영 및 하자보수의 유연성을 위한 모듈화 및 기존 소프트웨어와의 연계·호환성 확보
  - 개발된 모든 프로그램의 소스 및 이미지 제공
- **참가 자격**
  - ｢소프트웨어진흥법｣ 제2조에 의한 소프트웨어사업자(컴퓨터관련서비스사업, 1468)에 등록된 업체
  - ｢국가를 당사자로 하는 계약에 관한 법률｣ 제27조 및 동법 시행령 제76조에 결격사유가 없는 업체
  - ｢중소기업제품 구매촉진 및 판로지원에 관한 법률｣ 제2조제1호 및 제33조제1항의 중소기업자로서 동법 제8조의 경쟁입찰 참여자격과 “중‧소기업‧소상공인 확인서”를 소지한 업체
  - 대기업 및 중견기업인 소프트웨어사업자의 입찰 참여를 제한함
  - 직접생산증명서[정보시스템개발서비스(세부품명번호: 8111159901)]를 보유한 업체로 전자입찰서 제출 마감전일 발급된 것으로 유효기간 내에 있어야 함
  - 공동계약, 하도급 불가
- **평가**
  - 평가방식: 기술평가(90%)와 가격평가(10%)를 실시하여 종합평가점수로 평가
  - 기술·가격 배점: 기술평가 90점, 가격평가 10점 (경영상태 정량평가 5점 포함)
  - 협상적격 기준: 기술능력평가 점수가 배점한도(90점)의 85%(76.5점) 이상인 자
- **제출 마감**: 2024-09-19 12:00:00 (공고 정보 기준)
- **하자보수**
  - 무상하자 보수기간은 검수완료 후 1년(또는 사업종료 후 1년 이내)이며, S/W를 포함한 전체시스템에 대하여 발생한 장애는 무상조치함.
  - 하자담보책임기간 중 시스템 이상 발생 시 장애접수 후 2시간 이내 방문, 4시간 이내 장애조치 및 1일 24시간 365일 신속 대응이 이루어져야 함.
  - 하자보수 대상은 장비, 시스템 개발부분, 구입한 상용 소프트웨어 모두를 포함하며, DB 자료수정 및 갱신, 하드웨어 변경에 따른 S/W 설치, 성능 및 장애 처리·지원 등이 포함됨.
  - 하자보수 기간 종료 후 유상 전환 시 계약단가를 제시하며, 기능개선 및 재개발 등은 별도의 계약체결을 통해 유상으로 추진함.
- **공동수급·하도급**: 서면동의 없이는 제3자에게 양도하거나 사용할 수 없음(하도급 금지)